# Lymphoid annotation 

## Python modules

In [ ]:
import scanpy as sc
import anndata as ad

import scanorama
import scvi

from scipy import sparse
import pandas as pd
import numpy as np

import pickle

import seaborn as sb
import matplotlib.pyplot as plt

import os

## Settup rpy2 

In [ ]:
os.environ['R_HOME'] = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.1.0-FD20220623HSCT/lib/R'

In [ ]:
import rpy2.rinterface_lib.callbacks
import logging

from rpy2.robjects import pandas2ri
import anndata2ri

In [ ]:
%load_ext rpy2.ipython

## Settings

In [ ]:
# Working dir
os.chdir('/research/peer/fdeckert/FD20220623HSCT')

In [ ]:
# Warnings Python 
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Warnings R
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR) 

## R modules

In [78]:
%%R
library(Seurat)
library(dplyr)

## Plotting 

In [ ]:
# Scanpy figures
sc.set_figure_params(figsize=(5, 5), dpi_save=1200, fontsize=12, frameon=False, color_map='viridis', facecolor="white")

In [ ]:
# Matplot figures
plt.rcParams['figure.figsize']=(7, 7)

In [ ]:
# Load color 
import rpy2.robjects as robjects
color_load = robjects.r.source('plotting_global.R')
color = dict()
for i in range(len(color_load[0])):
    color[color_load[0].names[i]] = {key : color_load[0][i].rx2(key)[0] for key in color_load[0][i].names}

In [ ]:
# Set color function 
def set_color(adata, categories): 
    
    categories = [x for x in categories if x in list(adata.obs.columns)]

    for category in categories: 

        adata.obs[category] = pd.Series(adata.obs[category], dtype='category').cat.remove_unused_categories()
        
        keys = list(color[category].keys())
        keys = [x for x in keys if x in list(adata.obs[category])]        

        adata.obs[category] = adata.obs[category].cat.reorder_categories(keys)
        adata.uns[category+'_colors'] = np.array([color[category].get(key) for key in keys], dtype=object)
        
    return(adata)

# Processing functions 

## Dim reduction and cluster function 

In [ ]:
def dim_clust(adata, use_rep=None, n_neighbors=15, resolution=1, min_dist=0.5, metric='euclidean', file=None): 
    
    # Dim reduction and clustering on the scanorama corrected embedding 
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep=use_rep, metric=metric)
    if resolution is not None: 
        sc.tl.leiden(adata, resolution=resolution)
    
    sc.tl.umap(adata, min_dist=min_dist)
    
    # Write output 
    if file is not None: 
        adata.write_h5ad(file)
    
    return(adata)

## Differential expression analysis 

In [ ]:
def DEA(adata, key): 
    
    # Normalize
    sc.pp.normalize_total(adata, target_sum=1e6)
    sc.pp.log1p(adata)
    
    # DEA Wilcoxon 
    sc.tl.rank_genes_groups(adata, key, method='wilcoxon', key_added="wilcoxon", use_raw=False)
    sc.pl.rank_genes_groups(adata, n_genes=25, fontsize=12, sharey=False, key="wilcoxon", ncols=5)

# Annotate blood subset 

## Import data

In [ ]:
adata = sc.read_h5ad('data/object/subsets/round_3/lymphocytes_blood/scvi_con.h5ad')

In [ ]:
# Filter celltypist annotation 
adata.obs['celltypist_immun_low'] = adata.obs['celltypist_immun_low'].to_list()
adata.obs = adata.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata.n_obs/100), 'freq < 1 %') if x.name=='celltypist_immun_low' else x)

adata.obs['celltypist_immun_high'] = adata.obs['celltypist_immun_high'].to_list()
adata.obs = adata.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata.n_obs/100), 'freq < 1 %') if x.name=='celltypist_immun_high' else x)

## Gene modules

In [ ]:
tcr_genes_dict = {
    
    r'$TRC\alpha$': [i for i in adata.var_names if 'TRAC' in i],
    r'$TRC\beta$': [i for i in adata.var_names if 'TRBC' in i],
    r'$TRC\gamma$': [i for i in adata.var_names if 'TRGC' in i],
    r'$TRC\delta$': [i for i in adata.var_names if 'TRDC' in i],
    r'$TRV\alpha$': [i for i in adata.var_names if 'TRAV' in i],
    r'$TRV\beta$': [i for i in adata.var_names if 'TRBV' in i],
    r'$TRV\gamma$': [i for i in adata.var_names if 'TRGV' in i],
    r'$TRV\delta$': [i for i in adata.var_names if 'TRDV' in i]

}

In [ ]:
cell_type_genes_dict = {
    

    r'$TRC\alpha$': [i for i in adata.var_names if 'TRAC' in i],
    r'$TRC\beta$': [i for i in adata.var_names if 'TRBC' in i],
    r'$TRC\gamma$': [i for i in adata.var_names if 'TRGC' in i],
    r'$TRC\delta$': [i for i in adata.var_names if 'TRDC' in i],
    r'CD3': ['CD3E', 'CD3D', 'CD3G', 'CD247'],
    r'CD4': ['CD4'], 
    r'CD8': ['CD8A', 'CD8B'],
    r'Regulatory': ['CTLA4', 'IL2RA', 'FOXP3', 'TIGIT'], 
    r'Cytotoxic': ['KLRB1', 'GZMB', 'CCL5', 'NKG7', 'PRF1', 'GZMM'], # CD161:KLRB1
    r'NK': ['NCAM1', 'KLRD1', 'FCGR3A'], # CD56:NCAM1, CD94:KLRD1, CD16:FCGR3A | Possible TRGC/TRDC | CD3E has been found
    r'T_{inv}': ['ZBTB16'], # PZLF:ZBTB16, CD4+ or DN, four classes with some having restriceted TRAC/TRBC | Marker overlap with NK | Type I T_{inv} Vα24Jα18
    r'MAIT': ['SLC4A10', 'CEBPD'], # Vα7.2Jα33, TRAV1-2
    r'Residency': ['CREM', 'LMNA', 'EZR', 'ZFP36', 'CD69'], # CD69 distinguishes memory T cell in tissue from blood
    r'Naive': ['TCF7', 'SELL', 'CCR7'], # CD127:IL7R, CD62L:SELL
    r'Experienced': ['CD44'], #  
    r'Activated': ['CD38', 'FGFBP2', 'CD5'],  
    r'Exhaustion': ['PDCD1', 'TIGIT', 'HAVCR2'], # PDCD1:PD-1, HAVCR2:TIM3
    r'Cycling': ['PCNA', 'MKI67', 'TOP2A', 'CDK1']
    
}

## Dim reduction 

In [ ]:
adata = dim_clust(adata, use_rep='X_scvi', n_neighbors=30, resolution=2, min_dist=0.5, metric='euclidean', file='data/object/subsets/annotation/lymphocyte_blood.h5ad')

In [ ]:
sc.pl.umap(adata, color=['leiden', 'cell_type_leiden', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'cc_aa_alignment_size', 'celltypist_immun_high', 'celltypist_immun_low', 'msS_RNA'], legend_loc='right margin', ncols=4, wspace=0.3, size=50)

## Fine Leiden annotation 

In [ ]:
leiden_cell_type_fine = {
    
    '0': r'$T_{h, cm/naive}\ (0)$',
    '1': r'$T_{inv}\ (1)$',
    '2': r'$NK\ CD16-\ (2)$',
    '3': r'$NK\ CD16+\ (3)$',
    '4': r'$T_{c, em/effector}\ (4)$',
    '5': r'$T_{c, em/effector}\ (5)$',
    '6': r'$T_{c, em/effector}\ (6)$',
    '7': r'$T_{h, em/effector}\ (7)$',
    '8': r'$NK\ CD16+\ (8)$',
    '9': r'$Cycling\ (9)$',
    '10': r'$T_{reg}\ (10)$',
    '11': r'$T_{h, em/effector}\ (11)$',
    '12': r'$T_{h, em/effector}\ (12)$',
    '13': r'$T_{c, em/effector}\ (13)$',
    '14': r'$NK\ CD16+\ (14)$',
    '15': r'$T_{c, em/effector}\ (15)$',
    '16': r'$T_{c, em/effector}\ (16)$',
    '17': r'$T_{c, cm/naive}\ (17)$',
    '18': r'$NK\ CD16+\ (18)$',
    '19': r'$T_{h, cm/naive}\ (19)$',
    '20': r'$T_{c, em/effector}\ (20)$',
    '21': r'$T_{c, em/effector}\ (21)$',
    '22': r'$T_{c, em/effector}\ (22)$',
    '23': r'$T_{c, em/effector}\ (23)$',
    '24': r'$T_{c, em/effector}\ (24)$',
    '25': r'$T_{h, em/effector}\ (25)$',

}

In [ ]:
# Mapper 
leiden_cell_type_fine = pd.DataFrame({'leiden_cell_type_fine': adata.obs['leiden'].map(lambda x: leiden_cell_type_fine.get(x, x)).astype('category')})
adata.obs = adata.obs.merge(leiden_cell_type_fine, left_index=True, right_index=True, how='left')

In [ ]:
adata.obs['leiden_cell_type_fine'].cat.reorder_categories([
    
    
    r'$T_{h, em/effector}\ (7)$',
    r'$T_{h, em/effector}\ (11)$',
    r'$T_{h, em/effector}\ (12)$',
    r'$T_{h, em/effector}\ (25)$',
    
    r'$T_{h, cm/naive}\ (0)$',
    r'$T_{h, cm/naive}\ (19)$',
    
    r'$T_{reg}\ (10)$',
    
    r'$T_{c, em/effector}\ (4)$',
    r'$T_{c, em/effector}\ (5)$',
    r'$T_{c, em/effector}\ (6)$',
    r'$T_{c, em/effector}\ (13)$',
    r'$T_{c, em/effector}\ (15)$',
    r'$T_{c, em/effector}\ (16)$',
    r'$T_{c, em/effector}\ (20)$',
    r'$T_{c, em/effector}\ (21)$',
    r'$T_{c, em/effector}\ (22)$',
    r'$T_{c, em/effector}\ (23)$',
    r'$T_{c, em/effector}\ (24)$',
    
    r'$T_{c, cm/naive}\ (17)$',
    
    r'$T_{inv}\ (1)$',
    
    
    r'$NK\ CD16-\ (2)$',
    r'$NK\ CD16+\ (3)$',
    r'$NK\ CD16+\ (8)$',
    r'$NK\ CD16+\ (14)$',
    r'$NK\ CD16+\ (18)$',
    
    r'$Cycling\ (9)$'   
    
    
], inplace = True)

In [ ]:
dp = sc.pl.dotplot(adata, cell_type_genes_dict, groupby='leiden_cell_type_fine', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
dp = sc.pl.dotplot(adata, tcr_genes_dict, groupby='leiden_cell_type_fine', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
DEA(adata.copy(), key='leiden_cell_type_fine')

## Main Leiden annotation 

In [ ]:
leiden_cell_type_main = {
    
    '0': r'$T_{h, cm/naive}$',
    '1': r'$T_{inv}$',
    '2': r'$NK\ CD16-$',
    '3': r'$NK\ CD16+$',
    '4': r'$T_{c, em/effector}$',
    '5': r'$T_{c, em/effector}$',
    '6': r'$T_{c, em/effector}$',
    '7': r'$T_{h, em/effector}$',
    '8': r'$NK\ CD16+$',
    '9': r'$Cycling$',
    '10': r'$T_{reg}$',
    '11': r'$T_{h, em/effector}$',
    '12': r'$T_{h, em/effector}$',
    '13': r'$T_{c, em/effector}$',
    '14': r'$NK\ CD16+$',
    '15': r'$T_{c, em/effector}$',
    '16': r'$T_{c, em/effector}$',
    '17': r'$T_{c, cm/naive}$',
    '18': r'$NK\ CD16+$',
    '19': r'$T_{h, cm/naive}$',
    '20': r'$T_{c, em/effector}$',
    '21': r'$T_{c, em/effector}$',
    '22': r'$T_{c, em/effector}$',
    '23': r'$T_{c, em/effector}$',
    '24': r'$T_{c, em/effector}$',
    '25': r'$T_{h, em/effector}$',

}

In [ ]:
# Mapper 
leiden_cell_type_main = pd.DataFrame({'leiden_cell_type_main': adata.obs['leiden'].map(lambda x: leiden_cell_type_main.get(x, x)).astype('category')})
adata.obs = adata.obs.merge(leiden_cell_type_main, left_index=True, right_index=True, how='left')

In [ ]:
adata.obs['leiden_cell_type_main'].cat.reorder_categories([
    
    r'$T_{h, cm/naive}$', 
    r'$T_{h, em/effector}$',
    r'$T_{reg}$',
    
    r'$T_{c, cm/naive}$',
    r'$T_{c, em/effector}$',
     
    r'$T_{inv}$',
    
    r'$NK\ CD16-$',
    r'$NK\ CD16+$',
    
    r'$Cycling$' 
    
    
], inplace = True)

In [ ]:
dp = sc.pl.dotplot(adata, cell_type_genes_dict, groupby='leiden_cell_type_main', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
dp = sc.pl.dotplot(adata, tcr_genes_dict, groupby='leiden_cell_type_fine', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
DEA(adata.copy(), key='leiden_cell_type_main')

## Store annotation by cell id 

In [ ]:
leiden_cell_type_main_blood = adata.obs[['leiden', 'leiden_cell_type_main']]

# Annotate skin subset 

## Import data

In [ ]:
adata = sc.read_h5ad('data/object/subsets/round_3/lymphocytes_skin/scvi_con.h5ad')

In [ ]:
# Filter celltypist annotation 
adata.obs['celltypist_immun_low'] = adata.obs['celltypist_immun_low'].to_list()
adata.obs = adata.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata.n_obs/100), 'freq < 1 %') if x.name=='celltypist_immun_low' else x)

adata.obs['celltypist_immun_high'] = adata.obs['celltypist_immun_high'].to_list()
adata.obs = adata.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata.n_obs/100), 'freq < 1 %') if x.name=='celltypist_immun_high' else x)

## Gene modules

In [ ]:
tcr_genes_dict = {
    
    r'$TRC\alpha$': [i for i in adata.var_names if 'TRAC' in i],
    r'$TRC\beta$': [i for i in adata.var_names if 'TRBC' in i],
    r'$TRC\gamma$': [i for i in adata.var_names if 'TRGC' in i],
    r'$TRC\delta$': [i for i in adata.var_names if 'TRDC' in i],
    r'$TRV\alpha$': [i for i in adata.var_names if 'TRAV' in i],
    r'$TRV\beta$': [i for i in adata.var_names if 'TRBV' in i],
    r'$TRV\gamma$': [i for i in adata.var_names if 'TRGV' in i],
    r'$TRV\delta$': [i for i in adata.var_names if 'TRDV' in i]

}

In [ ]:
cell_type_genes_dict = {
    

    r'$TRC\alpha$': [i for i in adata.var_names if 'TRAC' in i],
    r'$TRC\beta$': [i for i in adata.var_names if 'TRBC' in i],
    r'$TRC\gamma$': [i for i in adata.var_names if 'TRGC' in i],
    r'$TRC\delta$': [i for i in adata.var_names if 'TRDC' in i],
    r'CD3': ['CD3E', 'CD3D', 'CD3G', 'CD247'],
    r'CD4': ['CD4'], 
    r'CD8': ['CD8A', 'CD8B'],
    r'Regulatory': ['CTLA4', 'IL2RA', 'FOXP3', 'TIGIT'], 
    r'Cytotoxic': ['KLRB1', 'GZMB', 'CCL5', 'NKG7', 'PRF1', 'GZMM'], # CD161:KLRB1
    r'NK': ['NCAM1', 'KLRD1', 'FCGR3A'], # CD56:NCAM1, CD94:KLRD1, CD16:FCGR3A | Possible TRGC/TRDC | CD3E has been found
    r'T_{inv}': ['ZBTB16'], # PZLF:ZBTB16, CD4+ or DN, four classes with some having restriceted TRAC/TRBC | Marker overlap with NK | Type I T_{inv} Vα24Jα18
    r'MAIT': ['CEBPD'], # Vα7.2Jα33, TRAV1-2, SLC4A10:NA
    r'Residency': ['CREM', 'LMNA', 'EZR', 'ZFP36', 'CD69'], # CD69 distinguishes memory T cell in tissue from blood
    r'Naive': ['TCF7', 'SELL', 'CCR7'], # CD127:IL7R, CD62L:SELL
    r'Experienced': ['CD44'], #  
    r'Activated': ['CD38', 'FGFBP2', 'CD5'],  
    r'Exhaustion': ['PDCD1', 'TIGIT', 'HAVCR2'], # PDCD1:PD-1, HAVCR2:TIM3
    r'Cycling': ['PCNA', 'MKI67', 'TOP2A', 'CDK1']
    
}

## Dim reduction 

In [ ]:
adata = dim_clust(adata, use_rep='X_scvi', n_neighbors=30, resolution=2, min_dist=0.5, metric='euclidean', file='data/object/subsets/annotation/lymphocyte_skin.h5ad')

In [ ]:
sc.tl.leiden(adata, resolution=0.25, restrict_to=('leiden', ['16']), key_added='leiden_restrict')

adata.obs['leiden'] = adata.obs['leiden_restrict']
del adata.obs['leiden_restrict']

In [ ]:
sc.pl.umap(adata, color=['leiden', 'cell_type_leiden', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'cc_aa_alignment_size', 'celltypist_immun_high', 'celltypist_immun_low', 'msS_RNA'], legend_loc='right margin', ncols=4, wspace=0.3, size=50)

## Fine Leiden annotation 

In [ ]:
leiden_cell_type_fine = {
    
    '0': r'$T_{h, memory/effector}\ (0)$',
    '1': r'$NK\ CD16-\ (1)$',
    '2': r'$T_{reg}\ (2)$',
    '3': r'$T_{c, memory/effector}\ (3)$',
    '4': r'$T_{h, memory/effector}\ (4)$',
    '5': r'$NK\ CD16-\ (5)$',
    '6': r'$NK\ CD16-\ (6)$',
    '7': r'$T_{c, memory/effector}\ (7)$',
    '8': r'$T_{c, memory/effector}\ (8)$',
    '9': r'$T_{c, memory/effector}\ (9)$',
    '10': r'$T_{c, memory/effector}\ (10)$',
    '11': r'$T_{h, memory/effector}\ (11)$',
    '12': r'$T_{c, memory/effector}\ (12)$',
    '13': r'$T_{inv}\ (13)$',
    '14': r'$T_{c, memory/effector}\ (14)$',
    '15': r'$T_{c, memory/effector}\ (15)$',
    '16,0': r'$T_{c, memory/effector}\ (16,0)$',
    '16,1': r'$NK\ CD16+\ (16,1)$',
    '17': r'$T_{h, memory/effector}\ (17)$',
    '18': r'$T_{c, memory/effector}\ (18)$',
    '19': r'$T_{c, memory/effector}\ (19)$',
    '20': r'$Cycling\ (20)$',
    '21': r'$T_{h, memory/effector}\ (21)$',
    '22': r'$T_{c, memory/effector}\ (22)$',
    '23': r'$T_{c, memory/effector}\ (23)$',
    '24': r'$T_{inv}\ (24)$',
    '25': r'$T_{c, memory/effector}\ (25)$'

}

In [ ]:
# Mapper 
leiden_cell_type_fine = pd.DataFrame({'leiden_cell_type_fine': adata.obs['leiden'].map(lambda x: leiden_cell_type_fine.get(x, x)).astype('category')})
adata.obs = adata.obs.merge(leiden_cell_type_fine, left_index=True, right_index=True, how='left')

In [ ]:
adata.obs['leiden_cell_type_fine'].cat.reorder_categories([
    
    r'$T_{h, memory/effector}\ (21)$',
    r'$T_{h, memory/effector}\ (0)$',
    r'$T_{h, memory/effector}\ (4)$',
    r'$T_{h, memory/effector}\ (11)$',
    r'$T_{h, memory/effector}\ (17)$',
    
    r'$T_{reg}\ (2)$',
    
    r'$T_{c, memory/effector}\ (12)$',
    r'$T_{c, memory/effector}\ (8)$',
    r'$T_{c, memory/effector}\ (7)$',
    r'$T_{c, memory/effector}\ (9)$',
    r'$T_{c, memory/effector}\ (10)$',
    r'$T_{c, memory/effector}\ (14)$',
    r'$T_{c, memory/effector}\ (15)$',
    r'$T_{c, memory/effector}\ (16,0)$',
    r'$T_{c, memory/effector}\ (18)$',
    r'$T_{c, memory/effector}\ (19)$',
    r'$T_{c, memory/effector}\ (23)$',
    r'$T_{c, memory/effector}\ (25)$',
    r'$T_{c, memory/effector}\ (3)$',
    r'$T_{c, memory/effector}\ (22)$',
    
    r'$T_{inv}\ (24)$',
    r'$T_{inv}\ (13)$',
    
    r'$NK\ CD16+\ (16,1)$', 
    r'$NK\ CD16-\ (5)$',
    r'$NK\ CD16-\ (1)$',
    r'$NK\ CD16-\ (6)$',
    
    r'$Cycling\ (20)$'
    
], inplace = True)

In [ ]:
dp = sc.pl.dotplot(adata, cell_type_genes_dict, groupby='leiden_cell_type_fine', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
dp = sc.pl.dotplot(adata, tcr_genes_dict, groupby='leiden_cell_type_fine', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
DEA(adata.copy(), key='leiden_cell_type_fine')

## Main Leiden annotation 

In [ ]:
leiden_cell_type_main = {
    
    '0': r'$T_{h, memory/effector}$',
    '1': r'$NK\ CD16-$',
    '2': r'$T_{reg}$',
    '3': r'$T_{c, memory/effector}$',
    '4': r'$T_{h, memory/effector}$',
    '5': r'$NK\ CD16-$',
    '6': r'$NK\ CD16-$',
    '7': r'$T_{c, memory/effector}$',
    '8': r'$T_{c, memory/effector}$',
    '9': r'$T_{c, memory/effector}$',
    '10': r'$T_{c, memory/effector}$',
    '11': r'$T_{h, memory/effector}$',
    '12': r'$T_{c, memory/effector}$',
    '13': r'$T_{inv}$',
    '14': r'$T_{c, memory/effector}$',
    '15': r'$T_{c, memory/effector}$',
    '16,0': r'$T_{c, memory/effector}$',
    '16,1': r'$NK\ CD16+$',
    '17': r'$T_{h, memory/effector}$',
    '18': r'$T_{c, memory/effector}$',
    '19': r'$T_{c, memory/effector}$',
    '20': r'$Cycling$',
    '21': r'$T_{h, memory/effector}$',
    '22': r'$T_{c, memory/effector}$',
    '23': r'$T_{c, memory/effector}$',
    '24': r'$T_{inv}$',
    '25': r'$T_{c, memory/effector}$'

}

In [ ]:
# Mapper 
leiden_cell_type_main = pd.DataFrame({'leiden_cell_type_main': adata.obs['leiden'].map(lambda x: leiden_cell_type_main.get(x, x)).astype('category')})
adata.obs = adata.obs.merge(leiden_cell_type_main, left_index=True, right_index=True, how='left')

In [ ]:
adata.obs['leiden_cell_type_main'].cat.reorder_categories([
    
    r'$T_{h, memory/effector}$',
    r'$T_{reg}$',
    
    r'$T_{c, memory/effector}$',
    
    r'$T_{inv}$', 
    
    r'$NK\ CD16+$', 
    r'$NK\ CD16-$',
    
    r'$Cycling$'
    
    
], inplace = True)

In [ ]:
dp = sc.pl.dotplot(adata, cell_type_genes_dict, groupby='leiden_cell_type_main', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
dp = sc.pl.dotplot(adata, tcr_genes_dict, groupby='leiden_cell_type_main', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
DEA(adata.copy(), key='leiden_cell_type_main')

## Store annotation by cell id 

In [ ]:
leiden_cell_type_main_skin = adata.obs[['leiden', 'leiden_cell_type_main']]

# Combine 

In [ ]:
adata = sc.read_h5ad('data/object/subsets/round_3/lymphocytes/scvi_con.h5ad')

In [ ]:
# Filter celltypist annotation 
adata.obs['celltypist_immun_low'] = adata.obs['celltypist_immun_low'].to_list()
adata.obs = adata.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata.n_obs/100), 'freq < 1 %') if x.name=='celltypist_immun_low' else x)

adata.obs['celltypist_immun_high'] = adata.obs['celltypist_immun_high'].to_list()
adata.obs = adata.obs.apply(lambda x: x.mask(x.map(x.value_counts())<int(adata.n_obs/100), 'freq < 1 %') if x.name=='celltypist_immun_high' else x)

## Gene modules

In [ ]:
tcr_genes_dict = {
    
    r'$TRC\alpha$': [i for i in adata.var_names if 'TRAC' in i],
    r'$TRC\beta$': [i for i in adata.var_names if 'TRBC' in i],
    r'$TRC\gamma$': [i for i in adata.var_names if 'TRGC' in i],
    r'$TRC\delta$': [i for i in adata.var_names if 'TRDC' in i],
    r'$TRV\alpha$': [i for i in adata.var_names if 'TRAV' in i],
    r'$TRV\beta$': [i for i in adata.var_names if 'TRBV' in i],
    r'$TRV\gamma$': [i for i in adata.var_names if 'TRGV' in i],
    r'$TRV\delta$': [i for i in adata.var_names if 'TRDV' in i]

}

In [ ]:
cell_type_genes_dict = {
    

    r'$TRC\alpha$': [i for i in adata.var_names if 'TRAC' in i],
    r'$TRC\beta$': [i for i in adata.var_names if 'TRBC' in i],
    r'$TRC\gamma$': [i for i in adata.var_names if 'TRGC' in i],
    r'$TRC\delta$': [i for i in adata.var_names if 'TRDC' in i],
    r'CD3': ['CD3E', 'CD3D', 'CD3G', 'CD247'],
    r'CD4': ['CD4'], 
    r'CD8': ['CD8A', 'CD8B'],
    r'Regulatory': ['CTLA4', 'IL2RA', 'FOXP3', 'TIGIT'], 
    r'Cytotoxic': ['KLRB1', 'GZMB', 'CCL5', 'NKG7', 'PRF1', 'GZMM'], # CD161:KLRB1
    r'NK': ['NCAM1', 'KLRD1', 'FCGR3A'], # CD56:NCAM1, CD94:KLRD1, CD16:FCGR3A | Possible TRGC/TRDC | CD3E has been found
    r'T_{inv}': ['ZBTB16'], # PZLF:ZBTB16, CD4+ or DN, four classes with some having restriceted TRAC/TRBC | Marker overlap with NK | Type I T_{inv} Vα24Jα18
    r'MAIT': ['SLC4A10', 'CEBPD'], # Vα7.2Jα33, TRAV1-2
    r'Residency': ['CREM', 'LMNA', 'EZR', 'ZFP36', 'CD69'], # CD69 distinguishes memory T cell in tissue from blood
    r'Naive': ['TCF7', 'SELL', 'CCR7'], # CD127:IL7R, CD62L:SELL
    r'Experienced': ['CD44'], #  
    r'Activated': ['CD38', 'FGFBP2', 'CD5'],  
    r'Exhaustion': ['PDCD1', 'TIGIT', 'HAVCR2'], # PDCD1:PD-1, HAVCR2:TIM3
    r'Cycling': ['PCNA', 'MKI67', 'TOP2A', 'CDK1']
    
}

## Add combined blood and skin annotation 

In [ ]:
leiden_cell_type_main = pd.concat([leiden_cell_type_main_blood, leiden_cell_type_main_skin])
leiden_cell_type_main['leiden_cell_type_main'] = leiden_cell_type_main['leiden_cell_type_main'].astype('category')

del adata.obs['leiden']
adata.obs = adata.obs.merge(leiden_cell_type_main, left_index=True, right_index=True, how='left')

In [ ]:
adata.obs['leiden_cell_type_main'].cat.reorder_categories([
    
    r'$T_{h, cm/naive}$', 
    r'$T_{h, em/effector}$',
    r'$T_{h, memory/effector}$',
    r'$T_{reg}$',
    
    r'$T_{c, cm/naive}$',
    r'$T_{c, em/effector}$',
    r'$T_{c, memory/effector}$', 
     
    r'$T_{inv}$',
    
    r'$NK\ CD16-$',
    r'$NK\ CD16+$',
    
    r'$Cycling$' 
    
    
], inplace = True)

## Dim reduction 

In [ ]:
adata = dim_clust(adata, use_rep='X_scvi', n_neighbors=30, resolution=None, min_dist=0.5, metric='euclidean', file='data/object/subsets/annotation/lymphocyte.h5ad')

In [ ]:
sc.pl.umap(adata, color=['leiden', 'leiden_cell_type_main', 'cc_phase_class', 'hto_demux_class', 'time_point', 'genotype_class', 'patient_id', 'has_ir', 'cc_aa_alignment_size', 'celltypist_immun_high', 'celltypist_immun_low', 'msS_RNA'], legend_loc='right margin', ncols=4, wspace=0.3, size=50)

In [ ]:
dp = sc.pl.dotplot(adata, cell_type_genes_dict, groupby='leiden_cell_type_main', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
dp = sc.pl.dotplot(adata, tcr_genes_dict, groupby='leiden_cell_type_main', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

## Set colors

In [ ]:
adata = set_color(adata, list(color.keys()))

In [ ]:
sc.pl.umap(adata, color=['leiden_cell_type_main'], legend_loc='right margin', ncols=4, wspace=0.3, size=50)

# Save result CSV

In [80]:
adata.obs.loc[:, ['leiden_cell_type_main']].rename(columns={'leiden_cell_type_main':'cell_type_leiden_subset'}).to_csv('result/annotation/sbusets/lymphocyte_p.csv')

# Save AnnData object

In [72]:
adata.write_h5ad('data/object/subsets/annotation/lymphocyte.h5ad')

## Save Seurat object

In [73]:
adata_r = adata.copy()

In [74]:
label_format = {
    
    r'$T_{h, cm/naive}$': "T['h, cm/naive']", 
    r'$T_{h, em/effector}$': "T['h, em/effector']",
    r'$T_{h, memory/effector}$': "T['h, memory/effector']",
    r'$T_{reg}$': "T['reg']",
    
    r'$T_{c, cm/naive}$': "T['c, cm/naive']",
    r'$T_{c, em/effector}$': "T['c, em/effector']",
    r'$T_{c, memory/effector}$': "T['c, memory/effector']", 
     
    r'$T_{inv}$': "T['inv']",
    
    r'$NK\ CD16-$': "NK*' CD16-'",
    r'$NK\ CD16+$': "NK*' CD16+'",
    
    r'$Cycling$': "Cycling" 
    
}

adata_r.obs['leiden_cell_type_main'] = adata_r.obs['leiden_cell_type_main'].map(lambda x: label_format.get(x, x)).astype('category')

In [75]:
label_format_order = [
    
    "T['h, cm/naive']", 
    "T['h, em/effector']",
    "T['h, memory/effector']",
    "T['reg']",
    
    "T['c, cm/naive']",
    "T['c, em/effector']",
    "T['c, memory/effector']", 
     
    "T['inv']",
    
    "NK*' CD16-'",
    "NK*' CD16+'",
    
    "Cycling" 
    
]

In [76]:
cnt = adata_r.X.T.todense()
cell_id = adata_r.obs_names
gene_id = adata_r.var_names
meta = adata_r.obs
umap = adata_r.obsm['X_umap']

In [79]:
%%R -i cnt -i cell_id -i gene_id -i meta -i label_format_order -i umap

colnames(cnt) <- cell_id
rownames(cnt) <- gene_id

so <- Seurat::CreateSeuratObject(cnt, meta.data=meta, project="lymphocyte", assay="RNA")

colnames(umap) <- c("UMAP_1", "UMAP_2")
rownames(umap) <- cell_id
so[["umap"]] <- CreateDimReducObject(embeddings=as.matrix(umap), key="UMAP_")

so$leiden_cell_type_main <- factor(so$leiden_cell_type_main, levels=label_format_order)
so$leiden_cell_type_fine <- factor(paste(so$leiden_cell_type_main, paste0("*' (", so$leiden, ")'")))

write.csv(so@meta.data[, "leiden_cell_type_main", drop=FALSE] %>% dplyr::rename(cell_type_leiden_subset="leiden_cell_type_main"), "result/annotation/sbusets/lymphocyte_r.csv")

saveRDS(so, 'data/object/subsets/annotation/lymphocyte.rds')